In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/01 06:24:22 WARN Utils: Your hostname, nabill-IdeaPad-3-14IML05 resolves to a loopback address: 127.0.1.1; using 192.168.11.111 instead (on interface wlp0s20f3)
26/10/01 06:24:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 06:24:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 06:24:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/nabill/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/nabill/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/nabill/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/nabill/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [3]:
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/nabill/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/nabill/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/nabill/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print(f"Data berhasil di ekstrak: ")
print(f" Transaksi : {df_transaksi.count()} baris")
print(f" Cabang : {df_cabang.count()} baris")
print(f" Menu : {df_menu.count()} baris")

Data berhasil di ekstrak: 
 Transaksi : 4000 baris
 Cabang : 3 baris
 Menu : 15 baris


In [6]:
df_gabungan_kopi = df_transaksi.join(df_cabang, on="cabang_id", how="inner")\
    .join(df_menu, on="menu_id", how="inner")

df_gabungan_kopi = df_gabungan_kopi.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan_kopi.select("trx_id", "nama_cabang", "nama_menu", "qty", "harga", "total_penjualan").show(10)

+------+--------------------+---------+---+-----+---------------+
|trx_id|         nama_cabang|nama_menu|qty|harga|total_penjualan|
+------+--------------------+---------+---+-----+---------------+
|   KN0|Kopi Nusantara Ma...|  Menu-14|  4|18000|          72000|
|   KN1|Kopi Nusantara Ma...|  Menu-11|  4|22000|          88000|
|   KN2|Kopi Nusantara Ma...|  Menu-11|  3|22000|          66000|
|   KN3|Kopi Nusantara Se...|  Menu-11|  1|22000|          22000|
|   KN4|Kopi Nusantara Yo...|  Menu-12|  4|35000|         140000|
|   KN5|Kopi Nusantara Yo...|  Menu-14|  3|18000|          54000|
|   KN6|Kopi Nusantara Se...|   Menu-4|  2|22000|          44000|
|   KN7|Kopi Nusantara Se...|   Menu-8|  4|30000|         120000|
|   KN8|Kopi Nusantara Se...|   Menu-9|  4|35000|         140000|
|   KN9|Kopi Nusantara Yo...|  Menu-10|  2|18000|          36000|
+------+--------------------+---------+---+-----+---------------+
only showing top 10 rows



In [7]:
penjualan_menu_cabang = df_gabungan_kopi.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan"))

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = penjualan_menu_cabang.withColumn("rank", row_number().over(window_cabang)) \
        .filter(col("rank") <= 2)

top2_menu.orderBy("nama_cabang", "rank").show()

+--------------------+---------+---------------+----+
|         nama_cabang|nama_menu|total_penjualan|rank|
+--------------------+---------+---------------+----+
|Kopi Nusantara Ma...|   Menu-1|        8505000|   1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|   2|
|Kopi Nusantara Se...|  Menu-13|        8750000|   1|
|Kopi Nusantara Se...|   Menu-9|        8715000|   2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|   1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|   2|
+--------------------+---------+---------------+----+



In [8]:
df_gabungan_kopi.createOrReplaceTempView("v_transaksi_kopi")

kinerja_barista = spark.sql('''
    SELECT kepala_barista, nama_cabang, SUM(total_penjualan) AS total_omset
    FROM v_transaksi_kopi
    GROUP BY kepala_barista, nama_cabang
    ORDER BY total_omset DESC
''')

kinerja_barista.show()

+--------------+--------------------+-----------+
|kepala_barista|         nama_cabang|total_omset|
+--------------+--------------------+-----------+
|         Nadia|Kopi Nusantara Se...|   90542000|
|         Nadia|Kopi Nusantara Ma...|   87138000|
|          Reza|Kopi Nusantara Yo...|   84645000|
+--------------+--------------------+-----------+



In [10]:
hdfs_output_tugas = "hdfs://localhost:9000/user/nabill/tugas7/processed/laporan"
df_gabungan_kopi.write.mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(hdfs_output_tugas)

print("Data berhasil disimpan ke HDFS")
!hdfs dfs -ls /user/nabill/tugas7/processed/laporan
df_verify_tugas = spark.read.parquet(hdfs_output_tugas)
print(f" verifikasi Pembacaan: {df_verify_tugas.count()} baris berhasil dimuat kembali")

Data berhasil disimpan ke HDFS


Found 4 items
-rw-r--r--   3 nabill supergroup          0 2026-10-01 06:58 /user/nabill/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:58 /user/nabill/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:58 /user/nabill/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:58 /user/nabill/tugas7/processed/laporan/kategori_menu=Non-Kopi
 verifikasi Pembacaan: 4000 baris berhasil dimuat kembali


In [11]:
ringkasan_kopi = df_gabungan_kopi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_omset"),
    count("trx_id").alias("total_transaksi"),
    avg("total_penjualan").alias("rata_rata_transaksi")).orderBy(col("total_omset").desc())

ringkasan_kopi.show()

+--------------------+----------+--------------+-----------+---------------+-------------------+
|         nama_cabang|      kota|kepala_barista|total_omset|total_transaksi|rata_rata_transaksi|
+--------------------+----------+--------------+-----------+---------------+-------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|   90542000|           1393|   64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|   87138000|           1302|  66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|   84645000|           1305|  64862.06896551724|
+--------------------+----------+--------------+-----------+---------------+-------------------+



Berdasarkan hasil analisis data pipeline end-toend terhadap performa penjualan jaringan kedai Kopi Nusantara, cabang kopi nusantara Semarang dibawa kepemimpinan kepala barista Nadia mencatatkan kinerja terbaik. cabang ini berhasil membukukan total omset tertinggi sebesar 90.542.000 dengan volume 1.393 transaksi serta nilai rata rata per transaksi mencapai 64.997

sebaliknya cabang kopi nusantara Yogyakarta yang dipimpin oleh Reza menunjukkan kinerja terlemah dengan nilai total 102.300.000 dan rata rata nilai transaksi terendah adalah 79.921. Berdasarkan temuan analisis window function pada bagian c, produk yang mendominasi penjualan di magelang adalah menu-15 dan menu-4 

untuk menaikkan performa cabang magelang, manajemen disarankan untuk melakukan strategi promodi gencar pada menu-25 dan menu -4. langkah nyata yang dapat diterapkan mencakup pembuatan paket bundling hemat dan juga program promosi jam sibuk. Selain itu kepala barista magelang perlu menerapkan teknik upselling kepada kasir untuk meningkatkan nilai rata rata transaksi konsumen agar mendekati pencapaian cabang yogyakarya